# 236. 二叉树的最近公共祖先

给定一个二叉树, 找到该树中两个指定节点的最近公共祖先。

百度百科中最近公共祖先的定义为：“对于有根树 T 的两个节点 p、q，最近公共祖先表示为一个节点 x，满足 x 是 p、q 的祖先且 x 的深度尽可能大（一个节点也可以是它自己的祖先）。”

示例1：

![1](https://assets.leetcode.com/uploads/2018/12/14/binarytree.png)

输入：root = [3,5,1,6,2,0,8,null,null,7,4], p = 5, q = 1

输出：3

解释：节点 5 和节点 1 的最近公共祖先是节点 3 。

示例2：

![2](https://assets.leetcode.com/uploads/2018/12/14/binarytree.png)

输入：root = [3,5,1,6,2,0,8,null,null,7,4], p = 5, q = 4

输出：5

解释：节点 5 和节点 4 的最近公共祖先是节点 5 。因为根据定义最近公共祖先节点可以为节点本身。

## 递归（DFS）

1. 递归函数与返回值

定义一个递归函数 `lowestCommonAncestor(root, p, q)`，它的功能是在以 root 为根节点的树中，寻找节点p、q的最近公共祖先。

返回值：
- 如果树里既有p又有q，就返回它们的最近公共祖先节点。
- 如果树里只有p，就返回p。
- 如果树里只有q，就返回q。
- 如果树里既没有p也没有q，就返回空。

2. 终止条件

有两个终止条件：
- 遇到空节点：说明这条路走到头了，返回空
- 遇到其中一个目标节点：这时候就不需要往下找了，因为父节点不会在子节点下面。

3. 这一层做什么

问左边、问右边、做决定。

去左子树里找找看：left = self.lowestCommonAncestor(root.left, p, q)；

去右子树里找找看：right = self.lowestCommonAncestor(root.right, p, q)；

结合左右结果做当前节点的决定：

情形 A：left 和 right 都有东西（都不是 null）。

说明左边找到了一个目标，右边也找到了一个目标。这就意味着 p 和 q 分别在我的左、右子树里。那我当前节点 root 就是它们的最近公共祖先！

👉 决定：返回 root。

情形 B：left 是 null，right 有东西。

说明左子树什么都没找到，目标全在右子树里。那么右子树返回的那个东西（不管是 p、q 还是它们在更低层的公共祖先），就是整棵树的答案。

👉 决定：返回 right。

情形 C：left 有东西，right 是 null。

同理，目标全在左子树里。

👉 决定：返回 left。

情形 D：left 和 right 都是 null。

说明这棵树底下干干净净，什么都没有。

👉 决定：返回 null（情况 B 和 C 的代码合并一下，就能包揽这种情况）。

In [ ]:
class Solution:
    def lowestCommonAncestor(self, root: 'TreeNode', p: 'TreeNode', q: 'TreeNode') -> 'TreeNode':
        # 要素二：终止条件
        if not root or root == p or root == q:
            return root
        
        # 要素三：单层逻辑 - 问左边
        left = self.lowestCommonAncestor(root.left, p, q)
        # 要素三：单层逻辑 - 问右边
        right = self.lowestCommonAncestor(root.right, p, q)
        
        # 要素三：单层逻辑 - 做决定
        if left and right:
            return root
            
        return left if left else right

## 🏢 场景记忆法：主管找员工

想象这棵二叉树是一家公司的组织架构，**当前节点（`root`）是一个部门主管**。
大老板派给你一个任务：**去你的部门里，把员工 `p` 和员工 `q` 给我找出来。**

作为主管，你不需要亲自去每个工位找，你只需要做一套标准的“找人与汇报”流程：

**第一步：看看自己是不是被找的人（终止条件）**
你低头一看工作牌，发现自己刚好就是 `p`（或者 `q`）。你立刻大喊一声：“不用找了，我就是！” 然后把自己汇报上去（`return root`）。如果你手下一个员工都没有（遇到了空节点），你就汇报：“我这儿没人”（`return null`）。

**第二步：安排左右副手去找（递归左右子树）**
如果你不是 `p` 也不是 `q`，你就把任务派给你的左副手和右副手：

* “左副手，你去你的部门里找找有没有 `p` 或 `q`，找到谁就把谁交上来。”（`left = ...`）
* “右副手，你也一样，去你的部门找。”（`right = ...`）

**第三步：根据副手的汇报，向大老板交差（单层逻辑）**
两个副手各自向你汇报了结果，此时你会有三种情况：

1. **左右逢源（`left` 和 `right` 都有人交上来）：** 你一拍大腿：“左边找到了一个，右边找到了一个！这说明 `p` 和 `q` 分别在我的两个副手下面，那我就是他们俩的**直接共同大领导（最近公共祖先）**啊！” 于是，你骄傲地向大老板汇报了**你自己**（`return root`）。
2. **一边倒（只有左边或者只有右边交了人上来）：** 左副手交上来了人，右副手说没找到。这说明要找的目标全在左副手的部门里。你不用管左副手交上来的是 `p`、是 `q`，还是某个底层的小主管，你只需要做一个“无情的传话筒”，把左副手交上来的东西，原封不动地递给大老板就行了（`return left`）。反之亦然。
3. **颗粒无收（两边都没找到）：** 两个副手都两手空空，你只能向大老板汇报：“我整个部门都没这俩人。”（`return null`）

